# Controlled Early vs Late Multimodal Fusion

This notebook implements the controlled experiment proposed for the LUAD–LUSC multimodal study.

It does five things:

1. **Regenerates WSI features safely** by training ABMIL using only the existing fusion training split; the held-out fusion test patients are never used to train the ABMIL model.
2. Creates two feature-budget settings:
   - **Original**: Clinical = all selected features (currently 44), RNA-Seq = 100, DNA methylation = 100, WSI = 100 PCA components.
   - **Equal40**: 40 features/components per modality.
3. Runs **early feature-level fusion** and **late decision-level fusion** on the same patient split.
4. Corrects the previous late-fusion implementation by **removing test-set min-max normalization** and tuning decision thresholds using out-of-fold training predictions only.
5. Runs modality ablations and saves result tables and figures.

> **Important:** Run this notebook after the Clinical, RNA-Seq, and DNA methylation preprocessing notebooks have produced their current feature CSV files. The notebook creates a new split-safe WSI representation from the HDF5 WSI bags, so the old `wsi_features.csv` is not used in the final controlled experiment.

## 1. Imports and global configuration

All paths follow the project structure already used in your preprocessing and fusion notebooks. Change only the values in this cell if your directory structure has moved.

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
import os

BASE_DIR = Path(
    os.environ.get("FUSION_REPO_DIR", str(Path.cwd()))
).expanduser().resolve()

if BASE_DIR.name in {"Preprocessing", "Training"}:
    BASE_DIR = BASE_DIR.parent

if not (BASE_DIR / "Training" / "07_controlled_early_late_fusion.ipynb").is_file():
    raise RuntimeError(
        "Start Jupyter from the repository directory or set FUSION_REPO_DIR."
    )
DATA_DIR = Path(
    os.environ.get("FUSION_DATA_DIR", str(BASE_DIR / "Data"))
).expanduser().resolve()
ANALYSIS_DIR = BASE_DIR
PREP_DIR = ANALYSIS_DIR / "Preprocessing"

LABELS_PATH = DATA_DIR / "multiomics_labels.tsv"
SPLIT_PATH = DATA_DIR / "train_test_split.csv"
BAGS_DIR = DATA_DIR / "bags_8k"

# Existing preprocessing outputs
CLINICAL_PATH = PREP_DIR / "clinical_features.csv"
RNA_PATH = PREP_DIR / "rna_selected_features.csv"
METH_PATH = PREP_DIR / "methylation_selected_features.csv"

# New controlled experiment outputs
CONTROL_DIR = ANALYSIS_DIR / "Controlled-Fusion"
FEATURE_DIR = CONTROL_DIR / "Features"
RESULT_DIR = CONTROL_DIR / "Results"
FIG_DIR = CONTROL_DIR / "Figures"
MODEL_DIR = CONTROL_DIR / "Models"

for d in [CONTROL_DIR, FEATURE_DIR, RESULT_DIR, FIG_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Experiment settings
# ---------------------------------------------------------
RANDOM_STATE = 42
CV_FOLDS = 10
N_JOBS = -1

EQUAL_BUDGET = 40
WSI_ORIGINAL_COMPONENTS = 100

RUN_ABMIL = True
FORCE_RETRAIN_ABMIL = False
RUN_ABLATIONS = True

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Configuration loaded.")
print(f"PREP_DIR    : {PREP_DIR}")
print(f"CONTROL_DIR : {CONTROL_DIR}")
print(f"Equal budget: {EQUAL_BUDGET} features per modality")

## 2. Load labels and the authoritative fusion split

The existing `train_test_split.csv` is treated as the **single authoritative split** for every modality and every fusion experiment.

- `train` patients may be used for feature learning, ABMIL training, cross-validation, and threshold selection.
- `test` patients are held out until final evaluation.

In [ ]:
def normalize_id(x):
    return str(x).strip().upper()[:12]

labels_df = pd.read_csv(LABELS_PATH, sep="\t")
labels_df["patient_id"] = labels_df["patient_id"].map(normalize_id)
labels_df["subtype_simple"] = labels_df["subtype_simple"].astype(str).str.strip()

labels = (
    labels_df
    .drop_duplicates("patient_id")
    .set_index("patient_id")["subtype_simple"]
    .map({"LUAD": 0, "LUSC": 1})
    .dropna()
    .astype(int)
)

split_df = pd.read_csv(SPLIT_PATH)
if "patient_id" not in split_df.columns:
    split_df = pd.read_csv(SPLIT_PATH, index_col=0).reset_index()
    split_df = split_df.rename(columns={split_df.columns[0]: "patient_id"})

split_df["patient_id"] = split_df["patient_id"].map(normalize_id)
split_df = split_df.drop_duplicates("patient_id").set_index("patient_id")
split_df = split_df.loc[split_df.index.intersection(labels.index)].copy()

fusion_train_ids = split_df.index[split_df["split"].astype(str).str.lower() == "train"].tolist()
fusion_test_ids = split_df.index[split_df["split"].astype(str).str.lower() == "test"].tolist()

print(f"Labels available : {len(labels)}")
print(f"Fusion train     : {len(fusion_train_ids)}")
print(f"Fusion test      : {len(fusion_test_ids)}")

print("\nTrain class counts:")
print(labels.loc[fusion_train_ids].value_counts().sort_index().rename(index={0: "LUAD", 1: "LUSC"}))

print("\nTest class counts:")
print(labels.loc[fusion_test_ids].value_counts().sort_index().rename(index={0: "LUAD", 1: "LUSC"}))

## 3. Load Clinical, RNA-Seq, and methylation feature matrices

Your preprocessing notebooks save selected features in ranked order:

- Clinical: ExtraTrees importance ranking.
- RNA-Seq: ExtraTrees ranking of selected DEGs.
- DNA methylation: absolute training-derived LUAD/LUSC beta-value difference.

Therefore, the **first 40 columns** of each saved matrix are the top 40 features under the existing selection rule.

In [ ]:
def load_feature_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    df = pd.read_csv(path, index_col=0)
    df.index = [normalize_id(i) for i in df.index]
    df = df[~df.index.duplicated(keep="first")]
    return df

clinical_full = load_feature_csv(CLINICAL_PATH)
rna_full = load_feature_csv(RNA_PATH)
meth_full = load_feature_csv(METH_PATH)

print("Existing preprocessing outputs:")
print(f"  Clinical       : {clinical_full.shape}")
print(f"  RNA-Seq        : {rna_full.shape}")
print(f"  DNA methylation: {meth_full.shape}")

for name, df in {
    "Clinical": clinical_full,
    "RNA-Seq": rna_full,
    "Methylation": meth_full,
}.items():
    if df.shape[1] < EQUAL_BUDGET:
        raise ValueError(f"{name} has only {df.shape[1]} features; cannot create Equal{EQUAL_BUDGET}.")

print("\nTop 5 retained features:")
print("Clinical:", clinical_full.columns[:5].tolist())
print("RNA-Seq :", rna_full.columns[:5].tolist())
print("Meth    :", meth_full.columns[:5].tolist())

# Part A — Split-safe WSI representation

## 4. ABMIL model definition

The updated WSI path is:

`1024-D UNI tile features -> gated ABMIL -> 512-D patient embedding`

The critical change is that ABMIL is trained **only on patients marked `train` in the fusion split**. A validation subset is carved from the fusion training patients for early stopping. The held-out fusion test set is never used for model selection or training.

In [ ]:
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FEAT_DIM = 1024
HIDDEN_DIM = 256
ATTN_DIM = 128
EMBED_DIM = 512
DROPOUT = 0.25
ABMIL_LR = 2e-4
WEIGHT_DECAY = 1e-5
ABMIL_EPOCHS = 30
PATIENCE = 7
VAL_FRACTION = 0.15

ABMIL_CHECKPOINT = MODEL_DIR / "abmil_fusion_split_safe.pt"
ABMIL_EMBEDDINGS = FEATURE_DIR / "wsi_embeddings_abmil_split_safe.pt"

print(f"Device: {DEVICE}")
print(f"ABMIL checkpoint : {ABMIL_CHECKPOINT}")
print(f"Embedding output : {ABMIL_EMBEDDINGS}")

class BagDataset(Dataset):
    def __init__(self, patient_ids, label_map, bags_dir):
        self.bags_dir = Path(bags_dir)
        self.label_map = label_map
        self.patient_ids = []
        missing = []
        for pid in patient_ids:
            if (self.bags_dir / f"{pid}.h5").exists():
                self.patient_ids.append(pid)
            else:
                missing.append(pid)
        if missing:
            print(f"Warning: {len(missing)} patients have no HDF5 WSI bag.")
            print("First missing IDs:", missing[:10])

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        path = self.bags_dir / f"{pid}.h5"
        with h5py.File(path, "r") as f:
            key = "features" if "features" in f else list(f.keys())[0]
            feats = torch.tensor(f[key][:], dtype=torch.float32)
        label = torch.tensor(self.label_map[pid], dtype=torch.long)
        return feats, label, pid


def collate_bags(batch):
    feats = [b[0] for b in batch]
    labels_batch = torch.stack([b[1] for b in batch])
    pids = [b[2] for b in batch]
    return feats, labels_batch, pids


class GatedAttentionMIL(nn.Module):
    def __init__(self, feat_dim, hidden_dim, attn_dim, embed_dim, dropout):
        super().__init__()
        self.fc1 = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.attn_V = nn.Linear(hidden_dim, attn_dim)
        self.attn_U = nn.Linear(hidden_dim, attn_dim)
        self.attn_w = nn.Linear(attn_dim, 1)
        self.fc2 = nn.Sequential(
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.fc1(x)
        a = self.attn_w(torch.tanh(self.attn_V(h)) * torch.sigmoid(self.attn_U(h)))
        a = F.softmax(a, dim=0)
        pooled = (a * h).sum(dim=0)
        emb = self.fc2(pooled)
        return emb, a.squeeze(1)


class ABMILClassifier(nn.Module):
    def __init__(self, feat_dim=FEAT_DIM, hidden_dim=HIDDEN_DIM, attn_dim=ATTN_DIM,
                 embed_dim=EMBED_DIM, dropout=DROPOUT, n_classes=2):
        super().__init__()
        self.encoder = GatedAttentionMIL(feat_dim, hidden_dim, attn_dim, embed_dim, dropout)
        self.classifier = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        emb, attn = self.encoder(x)
        logits = self.classifier(emb)
        return logits, emb, attn

## 5. Check WSI bag coverage

All patients in the authoritative fusion split should have an HDF5 WSI bag. The notebook stops rather than silently changing the cohort.

In [ ]:
# =========================================================
# 5. DEFINE THE COMPLETE FOUR-MODALITY COHORT
# =========================================================

# Patients with WSI HDF5 bags
wsi_bag_ids = {
    normalize_id(p.stem)
    for p in BAGS_DIR.glob("*.h5")
}

print("Available patients before multimodal intersection")
print(f"Split file        : {len(split_df):4d}")
print(f"Labels            : {len(labels):4d}")
print(f"Clinical features : {len(clinical_full):4d}")
print(f"RNA-Seq features  : {len(rna_full):4d}")
print(f"Methylation       : {len(meth_full):4d}")
print(f"WSI bags          : {len(wsi_bag_ids):4d}")


# Intersection across ALL required modalities
common_set = (
    set(split_df.index)
    & set(labels.index)
    & set(clinical_full.index)
    & set(rna_full.index)
    & set(meth_full.index)
    & wsi_bag_ids
)

# Preserve original split-file order
common_ids = [
    pid for pid in split_df.index
    if pid in common_set
]


# Rebuild train/test IDs for complete multimodal cohort
fusion_train_ids = [
    pid for pid in common_ids
    if str(split_df.loc[pid, "split"]).lower() == "train"
]

fusion_test_ids = [
    pid for pid in common_ids
    if str(split_df.loc[pid, "split"]).lower() == "test"
]

all_split_ids = fusion_train_ids + fusion_test_ids


print("\n" + "=" * 55)
print("FINAL COMPLETE MULTIMODAL COHORT")
print("=" * 55)

print(f"Total patients : {len(common_ids)}")
print(f"Train patients : {len(fusion_train_ids)}")
print(f"Test patients  : {len(fusion_test_ids)}")


print("\nTrain class distribution:")
print(
    labels.loc[fusion_train_ids]
    .value_counts()
    .sort_index()
    .rename(index={0: "LUAD", 1: "LUSC"})
)

print("\nTest class distribution:")
print(
    labels.loc[fusion_test_ids]
    .value_counts()
    .sort_index()
    .rename(index={0: "LUAD", 1: "LUSC"})
)


# Final sanity check
missing_bags = [
    pid for pid in common_ids
    if pid not in wsi_bag_ids
]

assert len(missing_bags) == 0, (
    f"Unexpected missing WSI bags in final cohort: {missing_bags}"
)

print("\n✓ Every patient in the final multimodal cohort has a WSI bag.")

## 6. ABMIL training helpers

Training proceeds in two stages:

1. Use an inner validation subset of the authoritative fusion training patients to select the training duration.
2. Reinitialize ABMIL and train on **all fusion training patients** for the selected number of epochs.

The final fusion test set remains untouched.

In [ ]:
label_map = labels.to_dict()


def make_loader(patient_ids, shuffle=False):
    ds = BagDataset(patient_ids, label_map, BAGS_DIR)
    return DataLoader(ds, batch_size=1, shuffle=shuffle, collate_fn=collate_bags, num_workers=0)


def run_abmil_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    y_true, y_prob = [], []
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for feats_list, y_batch, _ in loader:
            feats = feats_list[0].to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits, _, _ = model(feats)
            loss = criterion(logits.unsqueeze(0), y_batch)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            prob = torch.softmax(logits, dim=0)[1].item()
            y_true.append(int(y_batch.item()))
            y_prob.append(prob)

    y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
    return {
        "loss": total_loss / max(1, len(loader)),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "auc_roc": roc_auc_score(y_true, y_prob),
    }


def select_best_epoch(train_ids, seed=RANDOM_STATE):
    y_outer = labels.loc[train_ids].values
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=seed)
    inner_train_idx, val_idx = next(splitter.split(np.arange(len(train_ids)), y_outer))
    inner_train_ids = [train_ids[i] for i in inner_train_idx]
    val_ids = [train_ids[i] for i in val_idx]

    print(f"Inner train: {len(inner_train_ids)}")
    print(f"Validation : {len(val_ids)}")
    print(f"Final test : {len(fusion_test_ids)} (untouched)")

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = ABMILClassifier().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=ABMIL_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ABMIL_EPOCHS)
    criterion = nn.CrossEntropyLoss()

    train_loader = make_loader(inner_train_ids, shuffle=True)
    val_loader = make_loader(val_ids, shuffle=False)

    best_auc = -np.inf
    best_epoch = 1
    patience_count = 0
    history = []

    for epoch in range(1, ABMIL_EPOCHS + 1):
        tr = run_abmil_epoch(model, train_loader, criterion, optimizer=optimizer)
        va = run_abmil_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step()
        history.append({
            "epoch": epoch,
            "train_loss": tr["loss"],
            "train_auc": tr["auc_roc"],
            "val_loss": va["loss"],
            "val_auc": va["auc_roc"],
            "val_accuracy": va["accuracy"],
        })
        print(f"Epoch {epoch:02d} | train loss={tr['loss']:.4f} val AUC={va['auc_roc']:.4f} val acc={va['accuracy']:.4f}")

        if va["auc_roc"] > best_auc:
            best_auc = va["auc_roc"]
            best_epoch = epoch
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            print(f"Early stopping at epoch {epoch}; selected epoch={best_epoch}, val AUC={best_auc:.4f}")
            break

    return best_epoch, pd.DataFrame(history)


def train_final_abmil(train_ids, n_epochs, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = ABMILClassifier().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=ABMIL_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, n_epochs))
    criterion = nn.CrossEntropyLoss()
    train_loader = make_loader(train_ids, shuffle=True)

    for epoch in range(1, n_epochs + 1):
        metrics = run_abmil_epoch(model, train_loader, criterion, optimizer=optimizer)
        scheduler.step()
        print(f"Final ABMIL epoch {epoch:02d}/{n_epochs:02d} | loss={metrics['loss']:.4f} train AUC={metrics['auc_roc']:.4f}")

    return model

## 7. Train or load split-safe ABMIL and extract 512-D embeddings

If the checkpoint already exists, leave `FORCE_RETRAIN_ABMIL = False` to reuse it. The test-set score printed here is **evaluation only**; do not change hyperparameters based on it.

In [ ]:
if RUN_ABMIL:
    if ABMIL_CHECKPOINT.exists() and ABMIL_EMBEDDINGS.exists() and not FORCE_RETRAIN_ABMIL:
        print("Using existing split-safe ABMIL checkpoint and embeddings.")
        checkpoint = torch.load(ABMIL_CHECKPOINT, map_location=DEVICE, weights_only=False)
        model = ABMILClassifier().to(DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        best_epoch = checkpoint["selected_epoch"]
        embeddings = torch.load(ABMIL_EMBEDDINGS, map_location="cpu", weights_only=False)
    else:
        print("Selecting ABMIL training duration using fusion-train validation only...")
        best_epoch, abmil_history = select_best_epoch(fusion_train_ids)
        abmil_history.to_csv(RESULT_DIR / "abmil_validation_history.csv", index=False)
        print(f"\nSelected training duration: {best_epoch} epochs")

        print("\nTraining final ABMIL on all fusion training patients...")
        model = train_final_abmil(fusion_train_ids, n_epochs=best_epoch, seed=RANDOM_STATE)

        torch.save({
            "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
            "selected_epoch": best_epoch,
            "random_state": RANDOM_STATE,
            "fusion_train_ids": fusion_train_ids,
            "fusion_test_ids": fusion_test_ids,
        }, ABMIL_CHECKPOINT)

        model = model.to(DEVICE)
        model.eval()
        all_loader = make_loader(all_split_ids, shuffle=False)
        embeddings = {}
        with torch.no_grad():
            for feats_list, _, pids in all_loader:
                feats = feats_list[0].to(DEVICE)
                _, emb, _ = model(feats)
                embeddings[pids[0]] = emb.detach().cpu()
        torch.save(embeddings, ABMIL_EMBEDDINGS)
        print(f"Saved {len(embeddings)} patient embeddings.")

    criterion = nn.CrossEntropyLoss()
    test_loader = make_loader(fusion_test_ids, shuffle=False)
    test_metrics = run_abmil_epoch(model.to(DEVICE), test_loader, criterion, optimizer=None)

    print("\nSplit-safe ABMIL test performance (evaluation only):")
    for k, v in test_metrics.items():
        print(f"  {k:<10}: {v:.4f}")
else:
    if not ABMIL_EMBEDDINGS.exists():
        raise FileNotFoundError("RUN_ABMIL=False but no split-safe ABMIL embedding file exists.")
    embeddings = torch.load(ABMIL_EMBEDDINGS, map_location="cpu", weights_only=False)

print(f"\nEmbeddings available: {len(embeddings)}")

## 8. Fit WSI PCA on fusion training embeddings only

Two WSI matrices are generated from the **same split-safe 512-D ABMIL embeddings**:

- `WSI-100`: 100 PCA components for the Original feature-budget experiment.
- `WSI-40`: 40 PCA components for the Equal40 controlled experiment.

PCA is fitted on fusion training patients only and then applied unchanged to test patients.

In [ ]:
embedding_ids = [pid for pid in all_split_ids if pid in embeddings]

wsi_512 = pd.DataFrame(
    np.vstack([
        embeddings[pid].detach().cpu().numpy() if torch.is_tensor(embeddings[pid]) else np.asarray(embeddings[pid])
        for pid in embedding_ids
    ]),
    index=embedding_ids,
)
wsi_512.index.name = "patient_id"

if wsi_512.shape[1] != EMBED_DIM:
    raise ValueError(f"Expected {EMBED_DIM}-D ABMIL embeddings, got {wsi_512.shape[1]}.")

train_for_pca = [pid for pid in fusion_train_ids if pid in wsi_512.index]


def make_wsi_pca(n_components):
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    pca.fit(wsi_512.loc[train_for_pca])
    transformed = pca.transform(wsi_512)
    out = pd.DataFrame(
        transformed,
        index=wsi_512.index,
        columns=[f"wsi_pc_{i+1}" for i in range(n_components)],
    )
    return out, pca

wsi_100, pca_100 = make_wsi_pca(WSI_ORIGINAL_COMPONENTS)
wsi_40, pca_40 = make_wsi_pca(EQUAL_BUDGET)

WSI100_PATH = FEATURE_DIR / "wsi_features_split_safe_100.csv"
WSI40_PATH = FEATURE_DIR / "wsi_features_split_safe_40.csv"
wsi_100.to_csv(WSI100_PATH)
wsi_40.to_csv(WSI40_PATH)

print(f"512-D embeddings : {wsi_512.shape}")
print(f"WSI-100          : {wsi_100.shape} | variance={pca_100.explained_variance_ratio_.sum():.4%}")
print(f"WSI-40           : {wsi_40.shape} | variance={pca_40.explained_variance_ratio_.sum():.4%}")
print(f"\nSaved: {WSI100_PATH}")
print(f"Saved: {WSI40_PATH}")

# Part B — Controlled feature budgets

## 9. Build the Original and Equal40 modality sets

No raw preprocessing is repeated for Clinical, RNA-Seq, or DNA methylation. The Equal40 experiment takes the **top 40 already training-selected features** from each modality.

In [ ]:
original_modalities = {
    "Clinical": clinical_full.copy(),
    "RNA-Seq": rna_full.copy(),
    "Methylation": meth_full.copy(),
    "WSI": wsi_100.copy(),
}

equal40_modalities = {
    "Clinical": clinical_full.iloc[:, :EQUAL_BUDGET].copy(),
    "RNA-Seq": rna_full.iloc[:, :EQUAL_BUDGET].copy(),
    "Methylation": meth_full.iloc[:, :EQUAL_BUDGET].copy(),
    "WSI": wsi_40.copy(),
}

FEATURE_SETS = {
    "Original": original_modalities,
    "Equal40": equal40_modalities,
}

equal40_paths = {
    "Clinical": FEATURE_DIR / "clinical_features_fusion40.csv",
    "RNA-Seq": FEATURE_DIR / "rna_features_fusion40.csv",
    "Methylation": FEATURE_DIR / "methylation_features_fusion40.csv",
    "WSI": FEATURE_DIR / "wsi_features_fusion40.csv",
}

for modality, path in equal40_paths.items():
    equal40_modalities[modality].to_csv(path)

manifest_rows = []
for budget_name, modality_dict in FEATURE_SETS.items():
    print(f"\n{budget_name}")
    for modality, df in modality_dict.items():
        print(f"  {modality:<15}: {df.shape}")
        for rank, feature in enumerate(df.columns, start=1):
            manifest_rows.append({"budget": budget_name, "modality": modality, "rank": rank, "feature": feature})

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(FEATURE_DIR / "feature_budget_manifest.csv", index=False)
print("\nFeature manifest saved.")

## 10. Confirm patient alignment

All fusion comparisons must use the same patients, labels, and train/test designation.

In [ ]:
def ordered_common_ids(modality_dict, modality_names):
    common_set = set(split_df.index).intersection(labels.index)
    for m in modality_names:
        common_set &= set(modality_dict[m].index)
    return [pid for pid in split_df.index if pid in common_set]

for budget_name, modality_dict in FEATURE_SETS.items():
    full_ids = ordered_common_ids(modality_dict, ["Clinical", "RNA-Seq", "Methylation", "WSI"])
    local_split = split_df.loc[full_ids]
    n_train = (local_split["split"].astype(str).str.lower() == "train").sum()
    n_test = (local_split["split"].astype(str).str.lower() == "test").sum()
    print(f"{budget_name:<10}: {len(full_ids)} common patients (train={n_train}, test={n_test})")

# Part C — Fusion evaluation

## 11. Classifiers and modality combinations

The same six classical classifiers are used for both early and late fusion. The notebook evaluates the core three modality combinations and optional leave-one-modality-out ablations.

In [ ]:
CLASSIFIERS = {
    "LDA": LinearDiscriminantAnalysis(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "CART": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "NB": GaussianNB(),
    "SVM": SVC(kernel="rbf", C=1.0, probability=True, random_state=RANDOM_STATE),
    "LR": LogisticRegression(solver="saga", C=1.0, max_iter=2000, random_state=RANDOM_STATE),
}

UNIMODAL_COMBINATIONS = {
    "Unimodal-Clinical": ["Clinical"],
    "Unimodal-RNA-Seq": ["RNA-Seq"],
    "Unimodal-Methylation": ["Methylation"],
    "Unimodal-WSI": ["WSI"],
}

CORE_COMBINATIONS = {
    "Omics": ["RNA-Seq", "Methylation"],
    "Clinical+Omics": ["Clinical", "RNA-Seq", "Methylation"],
    "Full": ["Clinical", "RNA-Seq", "Methylation", "WSI"],
}

ABLATION_COMBINATIONS = {
    "Full-no-Clinical": ["RNA-Seq", "Methylation", "WSI"],
    "Full-no-RNA": ["Clinical", "Methylation", "WSI"],
    "Full-no-Methylation": ["Clinical", "RNA-Seq", "WSI"],
    "Full-no-WSI": ["Clinical", "RNA-Seq", "Methylation"],
}

EARLY_COMBINATIONS = {**UNIMODAL_COMBINATIONS, **CORE_COMBINATIONS}
LATE_COMBINATIONS = {**CORE_COMBINATIONS}
if RUN_ABLATIONS:
    EARLY_COMBINATIONS.update(ABLATION_COMBINATIONS)
    LATE_COMBINATIONS.update(ABLATION_COMBINATIONS)

FUSION_RULES = ["Maximum", "Sum", "Product"]

print("Classifiers:", list(CLASSIFIERS))
print("Early combinations:", list(EARLY_COMBINATIONS))
print("Late combinations :", list(LATE_COMBINATIONS))

## 12. Metrics and training-only threshold selection

A major correction relative to the previous late-fusion notebook is made here:

- **No min-max normalization is fitted on test predictions.**
- Decision thresholds are selected from **out-of-fold predictions generated using training patients only**.
- The selected threshold is frozen and applied once to the held-out test set.
- AUC-ROC remains threshold-independent.

In [ ]:
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "auc_roc": roc_auc_score(y_true, y_prob),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "y_pred": y_pred,
    }


def choose_threshold(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    candidates = np.unique(np.concatenate([
        np.linspace(0.05, 0.95, 181),
        np.quantile(y_prob, np.linspace(0.02, 0.98, 99)),
        np.array([0.5]),
    ]))
    best_threshold, best_f1, best_acc = 0.5, -np.inf, -np.inf
    for threshold in candidates:
        pred = (y_prob >= threshold).astype(int)
        f1 = f1_score(y_true, pred, average="macro")
        acc = accuracy_score(y_true, pred)
        if f1 > best_f1 + 1e-12 or (abs(f1 - best_f1) <= 1e-12 and acc > best_acc):
            best_threshold, best_f1, best_acc = float(threshold), float(f1), float(acc)
    return best_threshold


def fuse_probabilities(prob_list, rule):
    probs = np.column_stack(prob_list)
    probs = np.clip(probs, 1e-8, 1 - 1e-8)
    if rule == "Maximum":
        return probs.max(axis=1)
    if rule == "Sum":
        return probs.mean(axis=1)  # arithmetic mean = normalized sum rule
    if rule == "Product":
        return probs.prod(axis=1)
    raise ValueError(f"Unknown fusion rule: {rule}")


def make_cv():
    return StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

## 13. Helper for constructing early-fusion matrices

Feature names are prefixed by modality before concatenation to avoid accidental duplicate column names.

In [ ]:
def build_early_matrix(modality_dict, combo_modalities):
    common_ids = ordered_common_ids(modality_dict, combo_modalities)
    blocks = []
    for modality in combo_modalities:
        block = modality_dict[modality].loc[common_ids].copy()
        block.columns = [f"{modality}__{c}" for c in block.columns]
        blocks.append(block)
    X_df = pd.concat(blocks, axis=1)
    y_series = labels.loc[common_ids]
    split_local = split_df.loc[common_ids]
    train_mask = split_local["split"].astype(str).str.lower().eq("train").values
    test_mask = split_local["split"].astype(str).str.lower().eq("test").values
    return X_df, y_series, train_mask, test_mask, common_ids

## 14. Early feature-level fusion

For each classifier, the notebook generates training-only 10-fold out-of-fold probabilities, selects a threshold from those predictions, fits the classifier on all training patients, and evaluates once on the held-out test set.

In [ ]:
def evaluate_early_fusion(budget_name, modality_dict, combination_name, combination_modalities):
    X_df, y_series, train_mask, test_mask, common_ids = build_early_matrix(modality_dict, combination_modalities)
    X_train, X_test = X_df.values[train_mask], X_df.values[test_mask]
    y_train, y_test = y_series.values[train_mask], y_series.values[test_mask]
    rows = []

    print(f"\n[{budget_name}] {combination_name}: {X_df.shape[0]} patients, {X_df.shape[1]} features")

    for classifier_name, estimator in CLASSIFIERS.items():
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", clone(estimator))])
        oof_prob = cross_val_predict(pipe, X_train, y_train, cv=make_cv(), method="predict_proba", n_jobs=N_JOBS)[:, 1]
        threshold = choose_threshold(y_train, oof_prob)
        oof_metrics = compute_binary_metrics(y_train, oof_prob, threshold)

        pipe.fit(X_train, y_train)
        test_prob = pipe.predict_proba(X_test)[:, 1]
        test_metrics = compute_binary_metrics(y_test, test_prob, threshold)

        rows.append({
            "budget": budget_name,
            "strategy": "Early",
            "combination": combination_name,
            "modalities": "+".join(combination_modalities),
            "n_patients": len(common_ids),
            "n_features": X_df.shape[1],
            "n_train": int(train_mask.sum()),
            "n_test": int(test_mask.sum()),
            "classifier": classifier_name,
            "fusion_rule": "",
            "threshold": threshold,
            "oof_accuracy": oof_metrics["accuracy"],
            "oof_f1_macro": oof_metrics["f1_macro"],
            "oof_auc_roc": oof_metrics["auc_roc"],
            "oof_sensitivity": oof_metrics["sensitivity"],
            "oof_specificity": oof_metrics["specificity"],
            "test_accuracy": test_metrics["accuracy"],
            "test_f1_macro": test_metrics["f1_macro"],
            "test_auc_roc": test_metrics["auc_roc"],
            "test_sensitivity": test_metrics["sensitivity"],
            "test_specificity": test_metrics["specificity"],
        })

        print(f"  {classifier_name:<5} OOF AUC={oof_metrics['auc_roc']:.4f} test AUC={test_metrics['auc_roc']:.4f} test acc={test_metrics['accuracy']:.4f} thr={threshold:.3f}")

    return pd.DataFrame(rows)

## 15. Late decision-level fusion

One independent model is trained per modality, with the **same classifier family** used across all modalities in a given fused configuration. Modality-specific OOF probabilities are fused for training-only threshold selection, then final test probabilities are fused using the same rule.

In [ ]:
def evaluate_late_fusion(budget_name, modality_dict, combination_name, combination_modalities):
    common_ids = ordered_common_ids(modality_dict, combination_modalities)
    y_series = labels.loc[common_ids]
    split_local = split_df.loc[common_ids]
    train_mask = split_local["split"].astype(str).str.lower().eq("train").values
    test_mask = split_local["split"].astype(str).str.lower().eq("test").values
    y_all = y_series.values
    y_train, y_test = y_all[train_mask], y_all[test_mask]
    X_by_modality = {m: modality_dict[m].loc[common_ids].values for m in combination_modalities}
    rows = []

    print(f"\n[{budget_name}] {combination_name}: {len(common_ids)} patients, {len(combination_modalities)} modalities")

    for classifier_name, estimator in CLASSIFIERS.items():
        oof_prob_by_modality, test_prob_by_modality = [], []

        for modality in combination_modalities:
            X = X_by_modality[modality]
            X_train, X_test = X[train_mask], X[test_mask]
            pipe = Pipeline([("scaler", StandardScaler()), ("clf", clone(estimator))])
            oof_prob = cross_val_predict(pipe, X_train, y_train, cv=make_cv(), method="predict_proba", n_jobs=N_JOBS)[:, 1]
            pipe.fit(X_train, y_train)
            test_prob = pipe.predict_proba(X_test)[:, 1]
            oof_prob_by_modality.append(oof_prob)
            test_prob_by_modality.append(test_prob)

        for rule in FUSION_RULES:
            fused_oof = fuse_probabilities(oof_prob_by_modality, rule)
            threshold = choose_threshold(y_train, fused_oof)
            oof_metrics = compute_binary_metrics(y_train, fused_oof, threshold)
            fused_test = fuse_probabilities(test_prob_by_modality, rule)
            test_metrics = compute_binary_metrics(y_test, fused_test, threshold)

            rows.append({
                "budget": budget_name,
                "strategy": "Late",
                "combination": combination_name,
                "modalities": "+".join(combination_modalities),
                "n_patients": len(common_ids),
                "n_features": sum(modality_dict[m].shape[1] for m in combination_modalities),
                "n_train": int(train_mask.sum()),
                "n_test": int(test_mask.sum()),
                "classifier": classifier_name,
                "fusion_rule": rule,
                "threshold": threshold,
                "oof_accuracy": oof_metrics["accuracy"],
                "oof_f1_macro": oof_metrics["f1_macro"],
                "oof_auc_roc": oof_metrics["auc_roc"],
                "oof_sensitivity": oof_metrics["sensitivity"],
                "oof_specificity": oof_metrics["specificity"],
                "test_accuracy": test_metrics["accuracy"],
                "test_f1_macro": test_metrics["f1_macro"],
                "test_auc_roc": test_metrics["auc_roc"],
                "test_sensitivity": test_metrics["sensitivity"],
                "test_specificity": test_metrics["specificity"],
            })

        current = [r for r in rows if r["classifier"] == classifier_name]
        best_this_classifier = max(current, key=lambda r: r["oof_auc_roc"])
        print(f"  {classifier_name:<5} best rule={best_this_classifier['fusion_rule']:<7} OOF AUC={best_this_classifier['oof_auc_roc']:.4f} test AUC={best_this_classifier['test_auc_roc']:.4f} test acc={best_this_classifier['test_accuracy']:.4f}")

    return pd.DataFrame(rows)

# Part D — Run the controlled experiments

## 16. Run early fusion under both feature budgets

In [ ]:
early_result_frames = []
for budget_name, modality_dict in FEATURE_SETS.items():
    print("\n" + "=" * 80)
    print(f"EARLY FUSION — {budget_name}")
    print("=" * 80)
    for combination_name, combination_modalities in EARLY_COMBINATIONS.items():
        early_result_frames.append(
            evaluate_early_fusion(budget_name, modality_dict, combination_name, combination_modalities)
        )

early_results = pd.concat(early_result_frames, ignore_index=True)
EARLY_RESULTS_PATH = RESULT_DIR / "early_fusion_controlled.csv"
early_results.to_csv(EARLY_RESULTS_PATH, index=False)
print(f"\nSaved: {EARLY_RESULTS_PATH}")
print(f"Rows : {len(early_results)}")

## 17. Run late fusion under both feature budgets

In [ ]:
late_result_frames = []
for budget_name, modality_dict in FEATURE_SETS.items():
    print("\n" + "=" * 80)
    print(f"LATE FUSION — {budget_name}")
    print("=" * 80)
    for combination_name, combination_modalities in LATE_COMBINATIONS.items():
        late_result_frames.append(
            evaluate_late_fusion(budget_name, modality_dict, combination_name, combination_modalities)
        )

late_results = pd.concat(late_result_frames, ignore_index=True)
LATE_RESULTS_PATH = RESULT_DIR / "late_fusion_controlled.csv"
late_results.to_csv(LATE_RESULTS_PATH, index=False)
print(f"\nSaved: {LATE_RESULTS_PATH}")
print(f"Rows : {len(late_results)}")

# Part E — Fair model selection and comparison

## 18. Select the best model using training OOF AUC, not test AUC

This prevents choosing a classifier or fusion rule because it happened to perform well on the held-out test set.

In [ ]:
def select_best_by_oof(df):
    return (
        df.sort_values(
            ["budget", "combination", "oof_auc_roc", "oof_f1_macro"],
            ascending=[True, True, False, False],
        )
        .groupby(["budget", "combination"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

best_early = select_best_by_oof(early_results)
best_late = select_best_by_oof(late_results)

best_early.to_csv(RESULT_DIR / "best_early_by_oof.csv", index=False)
best_late.to_csv(RESULT_DIR / "best_late_by_oof.csv", index=False)

print("Best Early configurations:")
display(best_early[["budget", "combination", "classifier", "n_features", "oof_auc_roc", "test_accuracy", "test_auc_roc"]])

print("\nBest Late configurations:")
display(best_late[["budget", "combination", "classifier", "fusion_rule", "n_features", "oof_auc_roc", "test_accuracy", "test_auc_roc"]])

## 19. Direct early-vs-late comparison for matched modality combinations

The table below compares the **same modality combination and the same feature budget** on both sides.

In [ ]:
core_names = list(CORE_COMBINATIONS.keys())
early_core = best_early[best_early["combination"].isin(core_names)].copy()
late_core = best_late[best_late["combination"].isin(core_names)].copy()

comparison = early_core.merge(late_core, on=["budget", "combination"], suffixes=("_early", "_late"))
comparison["delta_accuracy_late_minus_early"] = comparison["test_accuracy_late"] - comparison["test_accuracy_early"]
comparison["delta_auc_late_minus_early"] = comparison["test_auc_roc_late"] - comparison["test_auc_roc_early"]

comparison_cols = [
    "budget", "combination",
    "classifier_early", "test_accuracy_early", "test_auc_roc_early",
    "classifier_late", "fusion_rule_late", "test_accuracy_late", "test_auc_roc_late",
    "delta_accuracy_late_minus_early", "delta_auc_late_minus_early",
]

comparison_table = comparison[comparison_cols].copy()
comparison_table.to_csv(RESULT_DIR / "early_vs_late_matched_comparison.csv", index=False)
display(comparison_table)

## 20. Modality-ablation summary

This analysis asks which modality contributes the most to the Full model. Negative change relative to Full means performance decreased when that modality was removed.

In [ ]:
if RUN_ABLATIONS:
    ablation_names = ["Full", "Full-no-Clinical", "Full-no-RNA", "Full-no-Methylation", "Full-no-WSI"]
    ablation_rows = []

    for budget in FEATURE_SETS:
        for strategy_name, df in [("Early", best_early), ("Late", best_late)]:
            block = df[(df["budget"] == budget) & (df["combination"].isin(ablation_names))].set_index("combination")
            if "Full" not in block.index:
                continue
            full_auc = block.loc["Full", "test_auc_roc"]
            full_acc = block.loc["Full", "test_accuracy"]

            for combination in ablation_names:
                if combination not in block.index:
                    continue
                row = block.loc[combination]
                ablation_rows.append({
                    "budget": budget,
                    "strategy": strategy_name,
                    "combination": combination,
                    "test_accuracy": row["test_accuracy"],
                    "test_auc_roc": row["test_auc_roc"],
                    "delta_accuracy_vs_full": row["test_accuracy"] - full_acc,
                    "delta_auc_vs_full": row["test_auc_roc"] - full_auc,
                })

    ablation_summary = pd.DataFrame(ablation_rows)
    ablation_summary.to_csv(RESULT_DIR / "ablation_summary.csv", index=False)
    display(ablation_summary)
else:
    print("RUN_ABLATIONS=False; ablation analysis skipped.")

# Part F — Figures

## 21. Matched early-vs-late AUC comparison

In [ ]:
for budget in FEATURE_SETS:
    plot_df = comparison_table[comparison_table["budget"] == budget].copy()
    if plot_df.empty:
        continue

    x = np.arange(len(plot_df))
    width = 0.36
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, plot_df["test_auc_roc_early"], width, label="Early")
    ax.bar(x + width/2, plot_df["test_auc_roc_late"], width, label="Late")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["combination"], rotation=15, ha="right")
    ax.set_ylabel("Held-out test AUC-ROC")
    ax.set_title(f"Early vs Late Fusion — {budget} Feature Budget")
    ax.legend()
    lower = min(plot_df["test_auc_roc_early"].min(), plot_df["test_auc_roc_late"].min())
    ax.set_ylim(max(0.0, lower - 0.03), 1.005)
    plt.tight_layout()
    out_path = FIG_DIR / f"early_vs_late_auc_{budget.lower()}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

## 22. Matched early-vs-late accuracy comparison

In [ ]:
for budget in FEATURE_SETS:
    plot_df = comparison_table[comparison_table["budget"] == budget].copy()
    if plot_df.empty:
        continue

    x = np.arange(len(plot_df))
    width = 0.36
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, plot_df["test_accuracy_early"], width, label="Early")
    ax.bar(x + width/2, plot_df["test_accuracy_late"], width, label="Late")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["combination"], rotation=15, ha="right")
    ax.set_ylabel("Held-out test accuracy")
    ax.set_title(f"Early vs Late Fusion — {budget} Feature Budget")
    ax.legend()
    lower = min(plot_df["test_accuracy_early"].min(), plot_df["test_accuracy_late"].min())
    ax.set_ylim(max(0.0, lower - 0.05), 1.005)
    plt.tight_layout()
    out_path = FIG_DIR / f"early_vs_late_accuracy_{budget.lower()}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

# Part G — Final WiML summary

## 23. Compact matched comparison table

Use the **Equal40** rows as the primary controlled analysis and the **Original** rows as a sensitivity analysis.

In [ ]:
final_report = comparison_table.copy()
for col in [
    "test_accuracy_early", "test_auc_roc_early",
    "test_accuracy_late", "test_auc_roc_late",
    "delta_accuracy_late_minus_early", "delta_auc_late_minus_early",
]:
    final_report[col] = final_report[col].round(4)

FINAL_REPORT_PATH = RESULT_DIR / "wiml_controlled_fusion_summary.csv"
final_report.to_csv(FINAL_REPORT_PATH, index=False)
print(f"Saved final summary: {FINAL_REPORT_PATH}")
display(final_report)

## 24. Interpretation checklist

Use this order when writing the paper or one-page WiML abstract:

1. **Primary result:** Equal40 matched comparison. This controls modality dimensionality.
2. **Sensitivity analysis:** Original feature budget (approximately 44/100/100/100).
3. **Model selection:** classifiers and late-fusion rules are selected using training OOF AUC, not test AUC.
4. **If Early ≈ Late:** report that fusion stage has little effect on discrimination once strong modality-specific representations are available.
5. **If Early > Late after Equal40:** this is stronger evidence that cross-modal feature interactions are useful.
6. **If Late > Early after Equal40:** this supports preserving modality-specific decision spaces.
7. Use the ablation table to discuss modality contribution separately from fusion strategy.
8. Do not claim external clinical generalization from TCGA alone.